In [1]:
import biom
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

ModuleNotFoundError: No module named 'biom'

In [ ]:
from skbio import TreeNode

In [3]:
pca = PCA(n_components=100, whiten=True)

### DNABERT embedding

In [9]:
import sys
import torch
import pandas as pd
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer


In [3]:
MODEL_NAME = "../models/dnabert2-117m"
FASTA_PATH = "../analysis/traits/data/feces_seq_16S_new.fasta"
RAW_OUTPUT = "dnabert2_16s_embedding.txt"
PCA_OUTPUT = "dnabert2_16s_embedding_reduced_100.txt"
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
def read_fasta(path):
    ids, sequences, seq = [], [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    sequences.append("".join(seq))
                    seq = []
                ids.append(line[1:].split()[0])
            else:
                seq.append(line.upper())
        if seq:
            sequences.append("".join(seq))
    return ids, sequences

ids, sequences = read_fasta(FASTA_PATH)


In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, local_files_only=True
)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    local_files_only=True,
    add_pooling_layer=False,
).to(device).eval()

model_module = sys.modules[model.__class__.__module__]
if hasattr(model_module, "flash_attn_qkvpacked_func"):
    model_module.flash_attn_qkvpacked_func = None

model


Some weights of the model checkpoint at ../models/dnabert2-117m were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(4096, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0): BertLayer(
        (attention): BertUnpadAttention(
          (self): BertUnpadSelfAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (mlp): BertGatedLinearUnitMLP(
          (gated_layers): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELU()
          (wo): Linear(in_features=3072, ou

In [6]:
def extract_embeddings(sequences, batch_size=8):
    embeddings = []

    for start in range(0, len(sequences), batch_size):
        batch = tokenizer(
            sequences[start:start + batch_size],
            add_special_tokens=True,
            padding=True,
            truncation=False,
            return_attention_mask=True,
            return_tensors="pt",
        )
        if "token_type_ids" not in batch:
            batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.inference_mode():
            output = model(**batch)
            hidden = output.last_hidden_state if hasattr(output, "last_hidden_state") else output[0]
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

        embeddings.append(pooled.cpu())

    return torch.cat(embeddings).numpy()


In [7]:
dnabert2_embed = pd.DataFrame(
    extract_embeddings(sequences, BATCH_SIZE),
    index=ids,
)
dnabert2_embed.to_csv(RAW_OUTPUT, sep=" ", header=False)
dnabert2_embed.shape


NameError: name 'pd' is not defined

In [ ]:
dnabert_pca = PCA(n_components=100, whiten=True)
dnabert2_embed_reduced = dnabert_pca.fit_transform(dnabert2_embed.values)
dnabert2_embed_reduced = pd.DataFrame(data=dnabert2_embed_reduced)
dnabert2_embed_reduced.index = dnabert2_embed.index.values
dnabert2_embed_reduced.to_csv(PCA_OUTPUT, header=None, sep=" ")


### PhyloE

In [6]:
tree = TreeNode.read("SSURefNR99_1200_slv_138_2_subset.tre")
dm = tree.tip_tip_distances()
dm = pd.DataFrame(data=dm.data, index=dm.ids, columns=dm.ids)
phy_embedding = pca.fit_transform(dm.values)
phy_embedding = pd.DataFrame(data=phy_embedding, index=dm.index.values)
phy_embedding.to_csv("phylo_embed_PCA_100.txt", sep=" ", header=None)